# Week 7: First-Order Logic: Unification

The practical is to implement unification in first-order logic. The notebook supports predicate arguments, variables, constants and nested function terms.

**Source:** `PPTs/Week7.pptx`.


### Unification rules
Two predicates unify only when their names and arities match. Variables can be substituted by constants, variables or terms. Substitutions must remain consistent; the occurs check prevents infinite terms such as `x = f(x)`.

## Program — First-order predicate unification

**Theory.** Unification finds a substitution that makes two first-order logic expressions identical. The predicate name and arity must match. Repeated variables must receive a consistent value, and the occurs check rejects circular bindings such as x=f(x).

**Objective.** Read two predicate expressions, find their most general unifier (MGU), or report why they cannot be unified.

**Libraries.** Python standard library (`ast`).

**Algorithm.**
1. Parse each predicate and its terms. 2. Require equal predicate names and arities. 3. Compare corresponding terms. 4. Bind variables consistently, recursively compare function terms, and apply the occurs check. 5. Print the MGU and unified expression.

**Complexity.** O(n) time and O(n) substitution space for n term nodes in a straightforward pass, excluding repeated substitution application.

The code below keeps `input()` so you can provide your own values. The final lines run a sample interaction through the same prompts; edit `SAMPLE_INPUTS` to try another case, or call the main function directly for a live terminal run.

**Expected output and explanation.** Expected bindings are x=Riya and y=Apple, yielding Eats(Riya,Apple).


In [1]:
from contextlib import contextmanager
import builtins

@contextmanager
def sample_input(values):
    original_input = builtins.input
    remaining = iter(map(str, values))
    def demo_input(prompt=''):
        print(prompt, end='')
        try:
            value = next(remaining)
        except StopIteration as exc:
            raise RuntimeError('Add more values to SAMPLE_INPUTS for this run.') from exc
        print(value)
        return value
    builtins.input = demo_input
    try:
        yield
    finally:
        builtins.input = original_input

import ast

def parse_term(node):
    if isinstance(node,ast.Name):
        return ('var',node.id) if node.id[0].islower() else ('const',node.id)
    if isinstance(node,ast.Constant) and isinstance(node.value,str): return ('const',node.value)
    if isinstance(node,ast.Call) and isinstance(node.func,ast.Name):
        return ('fn',node.func.id,tuple(parse_term(a) for a in node.args))
    raise ValueError('Terms must be variables, constants or function terms such as f(x).')

def parse_atom(text):
    node=ast.parse(text.strip(),mode='eval').body
    if not isinstance(node,ast.Call) or not isinstance(node.func,ast.Name): raise ValueError('Write an atom such as Eats(x, Apple).')
    return (node.func.id,tuple(parse_term(a) for a in node.args))

def apply(term,subs):
    if term[0]=='var':
        while term[0]=='var' and term[1] in subs: term=subs[term[1]]
        return apply(term,subs) if term[0]=='fn' else term
    if term[0]=='fn': return ('fn',term[1],tuple(apply(a,subs) for a in term[2]))
    return term

def occurs(name,term,subs):
    term=apply(term,subs)
    return (term[0]=='var' and term[1]==name) or (term[0]=='fn' and any(occurs(name,a,subs) for a in term[2]))

def unify_terms(left,right,subs=None):
    subs={} if subs is None else dict(subs); left=apply(left,subs); right=apply(right,subs)
    if left==right: return subs
    if left[0]=='var':
        if occurs(left[1],right,subs): return None
        subs[left[1]]=right; return subs
    if right[0]=='var': return unify_terms(right,left,subs)
    if left[0]=='const' or right[0]=='const': return None
    if left[1]!=right[1] or len(left[2])!=len(right[2]): return None
    for a,b in zip(left[2],right[2]):
        subs=unify_terms(a,b,subs)
        if subs is None: return None
    return subs

def unify_atoms(left,right):
    if left[0]!=right[0] or len(left[1])!=len(right[1]): return None
    subs={}
    for a,b in zip(left[1],right[1]):
        subs=unify_terms(a,b,subs)
        if subs is None: return None
    return subs

def show_term(term):
    if term[0]=='const': return term[1]
    if term[0]=='var': return term[1]
    return term[1]+'('+', '.join(show_term(a) for a in term[2])+')'

def main():
    left=input('First predicate expression: '); right=input('Second predicate expression: ')
    try: a,b=parse_atom(left),parse_atom(right)
    except (SyntaxError,ValueError) as exc: print('Invalid expression:',exc); return
    result=unify_atoms(a,b)
    if result is None: print('Unification fails: no consistent substitution exists.'); return
    print('Most general unifier:')
    if not result: print('{}')
    for name,value in result.items(): print(f'{name} = {show_term(apply(value,result))}')
    print('Unified expression:',a[0]+'('+', '.join(show_term(apply(t,result)) for t in a[1])+')')


SAMPLE_INPUTS = ['Eats(x, Apple)', 'Eats(Riya, y)']
with sample_input(SAMPLE_INPUTS):
    main()


First predicate expression: Eats(x, Apple)
Second predicate expression: Eats(Riya, y)
Most general unifier:
x = Riya
y = Apple
Unified expression: Eats(Riya, Apple)
